In [1]:
#Shishir Khanal
#Aug 24, 2026
#Lunar lander using REINFORCE algorithm using Tensorflow
#Actions: Do nothing, Fire Left Engine, Fire main Engine, Fire Right Engine
#Positive Score - Soft landing

In [2]:
import gym
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import tensorflow_probability as tfp
import matplotlib.pyplot as plt

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
2026-09-05 10:13:16.700383: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
#Policy Network
# Input layer -> 128 neuron hidden layer -> Output layer
#Adam optimizer
#ReLu activation where appropriate, Output unactivated
#Dont need input dims, tensorflow can infer dims
#Dont need lr, we will do model compile in Agent class
class PolicyNetwork(keras.Model):
    def __init__(self, n_actions, fc1_dims=256, fc2_dims=256, chkpt_dir='models/'):
        super(PolicyNetwork, self).__init__()
        self.fc1 = Dense(fc1_dims, activation='relu')
        self.fc2 = Dense(fc2_dims, activation='relu')
        self.pi = Dense(n_actions, activation='softmax')
    
    def call(self, state):
        x = self.fc1(state)
        x = self.fc2(x)
        x = self.pi(x)
        return x

In [4]:
#Agent: Constructor, action function and reward memory
#Agent needs gamma, policy Network, lists for memory of rewards, log probs
class Agent:
    def __init__(self, alpha=0.003, gamma=0.99, n_actions=4, fc1_dims=256, fc2_dims=256, chkpt_dir ='models/'):
        self.gamma = gamma
        self.state_memory = []
        self.reward_memory = []
        self.action_memory = []
        self.chkpt_dir = chkpt_dir
        self.policy = PolicyNetwork(n_actions=n_actions, fc1_dims=fc1_dims, fc2_dims=fc2_dims)
        self.policy.compile(optimizer=Adam(learning_rate=alpha))

    def save_models(self):
        self.policy.save_weights(self.chkpt_dir + 'reinforce.weights.h5')
        print('Model saved successfully...')

    def load_models(self):
        dummy_state = tf.zeros((1, 8), dtype=tf.float32)
        self.policy(dummy_state, training=False)
        self.policy.load_weights(self.chkpt_dir+'reinforce.weights.h5')
        print('Model loaded successfully...')

    def choose_action(self, observation):
        #Operation takes place outside of gradient tape
        #Recalculate pd in learn function
        state = tf.convert_to_tensor([observation], dtype=tf.float32)
        probabilities = self.policy(state)
        action_probs = tfp.distributions.Categorical(probs=probabilities)
        action = action_probs.sample()

        return action.numpy()[0]

    def store_transition(self,observation, action, reward):
        self.state_memory.append(observation)
        self.action_memory.append(action)
        self.reward_memory.append(reward)

    def learn(self):
        #No need to zero out gradients but conver to tensors
        actions = tf.convert_to_tensor(self.action_memory, dtype=tf.float32)
        rewards = np.array(self.reward_memory)
        #G_t = R_t+1 + gamma * R_t+2 + gamma**2 * R_t+3
        #G_t = sum from k=0 to k=T {gamma**k * R_t+k+1}
        G = np.zeros_like(self.reward_memory)
        for t in range(len(self.reward_memory)):
            G_sum = 0
            discount = 1
            for k in range(t, len(self.reward_memory)):
                G_sum += self.reward_memory[k] * discount
                discount *= self.gamma
            G[t] = G_sum

        with tf.GradientTape() as tape:
            loss = 0
            for idx, (g, state) in enumerate(zip(G, self.state_memory)):
                state = tf.convert_to_tensor([state], dtype=tf.float32)
                probs = self.policy(state)
                action_probs = tfp.distributions.Categorical(probs=probs)
                log_prob = action_probs.log_prob(actions[idx])
                loss += -g * tf.squeeze(log_prob)
        params = self.policy.trainable_variables
        grads = tape.gradient(loss, params)
        self.policy.optimizer.apply_gradients(zip(grads, params))
        
        self.state_memory = []
        self.action_memory = []
        self.reward_memory = []

In [5]:
def manage_memory():
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print(e)

In [6]:
def plot_learning_curve(scores, x, figure_file):
    running_avg = np.zeros(len(scores))
    for i in range(len(running_avg)):
        running_avg[i] = np.mean(scores[max(0, i-100):(i+1)])
    plt.plot(x, running_avg)
    plt.title('Running average of previous 100 scores')
    plt.show()
    plt.savefig(figure_file)

In [ ]:
if __name__ == '__main__':
    #manage_memory()
    env = gym.make('LunarLander-v2', render_mode="human")
    #env = gym.make('LunarLander-v2')
    n_games = 2000
    agent = Agent(gamma=0.99, alpha= 0.0005, n_actions=env.action_space.n)
    load_checkpoint = True
    best_score = -np.inf
    if load_checkpoint:
        n_games = 1
        agent.load_models()
        
    scores = []
    for i in range(n_games):
        done = False
        observation, info = env.reset()
        score = 0
        while not done:
            action = agent.choose_action(observation)
            observation_, reward, truncated, terminated, info = env.step(action)
            done = truncated or terminated
            score += reward
            agent.store_transition(observation, action, reward)
            observation = observation_ #current state = new state
            #env.render()
        if not load_checkpoint:
            agent.learn()
        scores.append(score)

        avg_score = np.mean(scores[-100:])
        if avg_score > best_score:
            if not load_checkpoint:
                agent.save_models()
            best_score = avg_score
        print('episode {} score {:.1f} average score {:.1f}'.format(i, score, avg_score))
    figure_file = 'plots/lunar_lander.png'
    x = [i+1 for i in range(n_games)]

In [ ]:
# if __name__ == '__main__':
#     #env = gym.make('LunarLander-v2', render_mode="human")
#     env = gym.make('LunarLander-v2')
#     n_games = 100
#     for i in range(n_games):
#         obs = env.reset()
#         score = 0
#         done = False
#         while not done:
#             action = env.action_space.sample()
#             obs_, reward, truncated, terminated, info = env.step(action)
#             done = truncated or terminated
#             score += reward
#             #env.render()
#         print('episode', i, 'score %.1f' % score)

In [ ]:
plot_learning_curve(scores, x, figure_file)